# Punctuate

Text punctuation is the process of adding or correcting punctuation marks in a text. Punctuation marks are symbols that help the reader understand the structure, meaning, and tone of the text. Some common punctuation marks are commas, periods, question marks, exclamation points, quotation marks, apostrophes, hyphens, dashes, colons, semicolons, parentheses, brackets, and ellipses.

Text punctuation is important for clear and effective communication, as it can avoid ambiguity, confusion, and misunderstanding. For example, compare the following sentences:

Let’s eat, Grandma. (The speaker is inviting their grandmother to eat with them.)
Let’s eat Grandma. (The speaker is suggesting to eat their grandmother.)
A woman without her man is nothing. (The speaker is implying that women need men to be complete.)
A woman: without her, man is nothing. (The speaker is implying that men need women to be complete.)
I love my parents, Beyoncé, and Jay-Z. (The speaker is listing four people they love.)
I love my parents, Beyoncé and Jay-Z. (The speaker is implying that their parents are Beyoncé and Jay-Z.)
As you can see, punctuation can change the meaning and tone of a sentence dramatically. Therefore, it is essential to use punctuation correctly and consistently in your writing.




## installing needed libraries

In [ ]:
!pip install transformers sentence_transformers
!pip install deepmultilingualpunctuation

## importing libraries

In [ ]:
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer
from transformers import pipeline
import pandas as pd
from collections import Counter
from deepmultilingualpunctuation import PunctuationModel
import re
import numpy as np
from sentence_transformers import SentenceTransformer, util


## loading pretrained model

In [ ]:
MODEL_LARGE = "oliverguhr/fullstop-punctuation-multilang-large"
MODEL_SONAR = "oliverguhr/fullstop-punctuation-multilingual-sonar-base"


TEST_FILE = "/content/drive/MyDrive/NLP/NLU/NLU/data/punctuation/punctuation.csv"

## loading pretrained model

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
df = pd.read_csv(TEST_FILE)
punc_large = PunctuationModel(MODEL_LARGE)
punc_sonar = PunctuationModel(MODEL_SONAR)


/usr/local/lib/python3.10/dist-packages/transformers/pipelines/token_classification.py:169: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="none"` instead.
  warnings.warn(


## processing function

In [ ]:



def get_most_sim(sentence, sentences):
    embeddings1 = model.encode(sentence, convert_to_tensor=True, show_progress_bar=False)
    embeddings2 = model.encode(sentences, convert_to_tensor=True, show_progress_bar=False)

    #Compute cosine-similarities
    cosine_scores = util.cos_sim(embeddings1, embeddings2)

    most_sim_id = np.argmax(cosine_scores.cpu()[0]).item()
    sim = max(cosine_scores.cpu()[0]).item()

    return sentences[most_sim_id], sim, most_sim_id


In [ ]:
df['text'].head()


0    In France voting has traditionally been a low-...
1    After officials verify the voter's identity th...
2    French electoral law rather strictly codifies ...
3    Since 1988 ballot boxes must be transparent so...
4    Candidates can send representatives to witness...
Name: text, dtype: object

In [ ]:

predicted_large = [];blue_large = []
predicted = [];blue = []
for text in df['text']:
  results = punc_sonar.restore_punctuation(text)
  predicted.append(results)
  _,sim, _ = get_most_sim(results, [text])
  blue.append(sim)

  results_large = punc_large.restore_punctuation(text)
  predicted_large.append(results_large)
  _,sim, _ = get_most_sim(results_large, [text])
  blue_large.append(sim)


In [ ]:
print(sum(blue_large)), print(sum(blue))

60.74820500612259
60.850284695625305


(None, None)

In [ ]:
  df['predicted-sonar']= predicted; df['all-MiniLM-L6-v2-sonar']= blue
  df['predicted-large']= predicted_large; df['all-MiniLM-L6-v2-large']= blue_large
  df.to_csv("/content/drive/MyDrive/NLP/NLU/NLU/data/punctuation/retults.csv", index=False)

## Create data


In [ ]:
# TEST_FILE = "/content/drive/MyDrive/NLP/NLU/NLU/data/punctuation/punctuation.tsv"
# df = pd.read_csv(TEST_FILE, sep="\t")
# newdf = df[['lang_id', 'src_text',  'tgt_text']].iloc[300:360]
# newdf.to_csv("/content/drive/MyDrive/NLP/NLU/NLU/data/punctuation.csv",)
# clean_text = re.sub (r'[,|.|:]+', '', text)